# SUPER FANCY BUT ACCURATELY DESCRIPTIVE TITLE
### DATE - *Tyndale Stutzman*
---

## Introduction:


## Table of Contents


In [1]:
# Save first; after running, reload/revert this notebook from disk without saving.
# This edits the file, not the live editor. Update the filename if you rename it.
import json, re
from pathlib import Path
from html import escape, unescape
from urllib.parse import unquote
from uuid import uuid4

path = Path("template.ipynb")
if not path.is_file():
    path = Path("notebooks") / path
book = json.loads(path.read_text())
cells = book["cells"]
indices = [i for i, c in enumerate(cells) if "toc-generator" in c.get("metadata", {}).get("tags", [])]
if len(indices) != 1 or indices[0] == 0:
    raise ValueError("Keep one generator cell, directly below the TOC.")
index = indices[0]
if cells[index - 1]["cell_type"] != "markdown":
    raise ValueError("The cell above must be the Markdown TOC.")
toc = "".join(cells[index - 1]["source"])
entries = re.findall(r"^([ \t]*)(?:\d+[.)]|[-+*])\s*\[(.+?)\]\(#([^\s)]+)\)[ \t]*$", toc, re.M)
entries = [(len(indent.expandtabs(4)), title, unquote(anchor)) for indent, title, anchor in entries]
levels = sorted({indent for indent, _, _ in entries})
if not entries or len(levels) > 5 or len({a for _, _, a in entries}) != len(entries):
    raise ValueError("Use unique TOC anchors and at most five indentation levels.")
existing = {unescape(anchor): c for c in cells if c["cell_type"] == "markdown"
            for anchor in re.findall(r'<a\b[^>]*\bid=["\']([^"\']+)["\']', "".join(c["source"]))}
added = 0
for position, (indent, title, anchor) in enumerate(entries):
    if anchor in existing:
        continue
    following = next((existing[a] for _, _, a in entries[position + 1:] if a in existing and cells.index(existing[a]) > index), None)
    insertion = cells.index(following) if following is not None else len(cells)
    heading = f'<a id="{escape(anchor, quote=True)}"></a>\n{"#" * (2 + levels.index(indent))} {title}\n'
    cell = {"cell_type": "markdown", "id": uuid4().hex[:8],
            "metadata": {"tags": ["toc-section"]}, "source": heading.splitlines(keepends=True)}
    cells.insert(insertion, cell)
    existing[anchor] = cell
    added += 1
if added:
    path.write_text(json.dumps(book, indent=1, ensure_ascii=False) + "\n")
print(f"Added {added} section(s). Reload/revert from disk; do not save the old editor contents.")


Added 0 section(s). Reload/revert from disk; do not save the old editor contents.
